# Ticker List Analysis

This notebook uses only ticker CSV files found in `dataset/stocks` and visualizes ticker and data-quality insights from that source.


## Cell 1: Imports
Loads the core Python libraries used throughout the notebook for paths, data frames, and plotting.


In [1]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


## Cell 2: Analysis Settings
Defines notebook-level parameters used for previews and top/bottom ranking plots.


In [2]:
TOP_N = 20
MAX_PREVIEW_ROWS = 10


## Cell 3: Project Root Helpers
Finds the project root, resolves the `dataset/stocks` source path, and defines small shared table helpers.


In [3]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "dataset").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing dataset/")


def resolve_stocks_dir(project_root: Path) -> Path:
    stocks_dir = project_root / "dataset" / "stocks"
    if not stocks_dir.exists():
        raise FileNotFoundError(f"Required source directory not found: {stocks_dir}")
    return stocks_dir


def load_source_tickers(stocks_dir: Path) -> tuple[list[str], list[Path]]:
    price_paths = sorted(stocks_dir.glob("*.csv"))
    tickers = sorted({path.stem.upper() for path in price_paths})
    return tickers, price_paths


def count_table(series: pd.Series, index_name: str, count_name: str = "count") -> pd.DataFrame:
    return series.value_counts().rename_axis(index_name).to_frame(count_name)


PROJECT_ROOT = find_project_root(Path.cwd())
print(f"Project root: {PROJECT_ROOT}")


FileNotFoundError: Could not locate project root containing dataset/

## Cell 4: Data Parsing And Metrics Helpers
Defines reusable functions to clean per-ticker CSVs and compute quality/market metrics (rows, returns, volatility, volume).


In [ ]:
NUMERIC_COLUMNS = ("Adj Close", "Close", "High", "Low", "Open", "Volume")
STATUS_OK = "ok"
STATUS_EMPTY = "empty"
STATUS_UNPARSED = "unparsed"
RESULT_COLUMNS = [
    "ticker",
    "rows",
    "start_date",
    "end_date",
    "price_start",
    "price_end",
    "total_return_pct",
    "daily_volatility_pct",
    "avg_volume",
]
PREVIEW_COLUMNS = [
    "ticker",
    "status",
    "rows",
    "start_date",
    "end_date",
    "price_start",
    "price_end",
    "total_return_pct",
    "daily_volatility_pct",
    "avg_volume",
]
RANKED_FIGSIZE = (20, 7)


def _base_summary(ticker: str, status: str, rows: int = 0) -> dict:
    return {
        "ticker": ticker,
        "status": status,
        "rows": int(rows),
        "start_date": None,
        "end_date": None,
        "total_return_pct": np.nan,
        "daily_volatility_pct": np.nan,
        "avg_volume": np.nan,
        "price_start": np.nan,
        "price_end": np.nan,
        "price_range": np.nan,
        "range_pct": np.nan,
        "volume_range": np.nan,
        "volume_range_pct": np.nan,
    }


def _series_range_metrics(series: pd.Series) -> tuple[float, float]:
    clean = series.dropna()
    if clean.empty:
        return np.nan, np.nan

    max_val = float(clean.max())
    min_val = float(clean.min())
    value_range = max_val - min_val
    pct = float(value_range / min_val * 100) if min_val > 0 else np.nan
    return value_range, pct


def _pct_change(first: float, last: float) -> float:
    if first in (0.0, -0.0) or not np.isfinite(first) or not np.isfinite(last):
        return np.nan
    return float((last / first - 1.0) * 100)


def _price_span_metrics(high: pd.Series, low: pd.Series) -> tuple[float, float]:
    high_clean = high.dropna()
    low_clean = low.dropna()
    if high_clean.empty or low_clean.empty:
        return np.nan, np.nan

    max_high = float(high_clean.max())
    min_low = float(low_clean.min())
    price_range = max_high - min_low
    range_pct = float(price_range / min_low * 100) if min_low > 0 else np.nan
    return price_range, range_pct


def load_local_price_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if df.empty:
        return df

    parsed_first_date = pd.to_datetime(df.iloc[0].get("Date"), errors="coerce")
    if pd.isna(parsed_first_date):
        # Some yfinance exports include an extra symbol row after the header.
        df = df.iloc[1:].copy()

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    for col in NUMERIC_COLUMNS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)


def summarize_price_file(path: Path) -> dict:
    ticker = path.stem.upper()
    try:
        df = load_local_price_csv(path)
    except Exception as exc:
        return _base_summary(ticker, f"error: {exc}")

    if df.empty:
        return _base_summary(ticker, STATUS_EMPTY)

    close = df["Close"].dropna() if "Close" in df.columns else pd.Series(dtype=float)
    if close.empty:
        price_start = np.nan
        price_end = np.nan
        total_return_pct = np.nan
        daily_volatility_pct = np.nan
    else:
        price_start = float(close.iloc[0])
        price_end = float(close.iloc[-1])

        if len(close) >= 2:
            # Percent change from first close in the file to last close in the file.
            total_return_pct = _pct_change(price_start, price_end)
            daily_volatility_pct = float(close.pct_change().std() * 100)
        else:
            total_return_pct = np.nan
            daily_volatility_pct = np.nan

    avg_volume = float(df["Volume"].dropna().mean()) if "Volume" in df.columns else np.nan

    if "Volume" in df.columns:
        volume_range, volume_range_pct = _series_range_metrics(df["Volume"])
    else:
        volume_range, volume_range_pct = np.nan, np.nan

    if "High" in df.columns and "Low" in df.columns:
        price_range, range_pct = _price_span_metrics(df["High"], df["Low"])
    else:
        price_range, range_pct = np.nan, np.nan

    summary = _base_summary(ticker, STATUS_OK, rows=len(df))
    summary.update(
        {
            "start_date": df["Date"].min().date(),
            "end_date": df["Date"].max().date(),
            "price_start": price_start,
            "price_end": price_end,
            "total_return_pct": total_return_pct,
            "daily_volatility_pct": daily_volatility_pct,
            "avg_volume": avg_volume,
            "price_range": price_range,
            "range_pct": range_pct,
            "volume_range": volume_range,
            "volume_range_pct": volume_range_pct,
        }
    )
    return summary


def build_quality_frame(paths: list[Path]) -> pd.DataFrame:
    return pd.DataFrame([summarize_price_file(path) for path in paths])


def build_analysis_frame(tickers: pd.DataFrame, quality: pd.DataFrame) -> pd.DataFrame:
    analysis = tickers.merge(quality, on="ticker", how="left")
    analysis["status"] = analysis["status"].fillna(STATUS_UNPARSED)
    analysis["rows"] = analysis["rows"].fillna(0).astype(int)
    return analysis


def _plot_hist(series: pd.Series, ax, title: str, xlabel: str, color: str, alpha: float = 0.85) -> None:
    series.dropna().plot(kind="hist", bins=30, ax=ax, color=color, alpha=alpha)
    ax.set_title(title)
    ax.set_xlabel(xlabel)


def _plot_ranked_bars(ax, labels: pd.Series, values: pd.Series, title: str, ylabel: str, color: str) -> None:
    ax.bar(labels, values, color=color)
    ax.set_title(title)
    ax.set_xlabel("Ticker")
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=70)


def plot_overview_charts(ok_df: pd.DataFrame) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))

    _plot_hist(ok_df["rows"], axes[0, 0], "Distribution of Rows per Ticker", "Rows", "#264653")

    returns = ok_df["total_return_pct"].dropna()
    axes[0, 1].set_title("Total Return Distribution (%)")
    axes[0, 1].set_xlabel("Total Return %")
    axes[0, 1].set_ylabel("Frequency")

    if returns.empty:
        axes[0, 1].text(0.5, 0.5, "No total return data", ha="center", va="center", transform=axes[0, 1].transAxes)
    else:
        bins = np.histogram_bin_edges(returns, bins=30)
        counts, edges = np.histogram(returns, bins=bins)
        left_edges = edges[:-1]
        widths = np.diff(edges)
        centers = left_edges + widths / 2

        neg_mask = centers < 0
        pos_mask = ~neg_mask

        if neg_mask.any():
            axes[0, 1].bar(
                left_edges[neg_mask],
                counts[neg_mask],
                width=widths[neg_mask],
                align="edge",
                color="#c1121f",
                alpha=0.8,
                label="Negative",
            )
        if pos_mask.any():
            axes[0, 1].bar(
                left_edges[pos_mask],
                counts[pos_mask],
                width=widths[pos_mask],
                align="edge",
                color="#2a9d8f",
                alpha=0.8,
                label="Positive",
            )

        axes[0, 1].axvline(0, color="#555555", linestyle="--", linewidth=1)
        axes[0, 1].legend()

    volatility = ok_df["daily_volatility_pct"].dropna()
    axes[1, 0].set_title("Daily Volatility Distribution (%)")
    axes[1, 0].set_xlabel("Daily Volatility %")
    axes[1, 0].set_ylabel("Frequency")

    if volatility.empty:
        axes[1, 0].text(0.5, 0.5, "No daily volatility data", ha="center", va="center", transform=axes[1, 0].transAxes)
    else:
        q_low, q_high = volatility.quantile([0.01, 0.99])
        use_trim = np.isfinite(q_low) and np.isfinite(q_high) and q_high > q_low

        if use_trim:
            focus = volatility[(volatility >= q_low) & (volatility <= q_high)]
            if focus.empty:
                focus = volatility
        else:
            focus = volatility

        vmin = float(focus.min())
        vmax = float(focus.max())
        if np.isclose(vmin, vmax):
            left = vmin * 0.9 if vmin > 0 else vmin - 0.1
            right = vmax * 1.1 if vmax > 0 else vmax + 0.1
            bins = np.array([left, right])
        else:
            bins = np.linspace(vmin, vmax, 30)

        axes[1, 0].hist(focus, bins=bins, color="#e9c46a", alpha=0.9)

        if use_trim:
            pad = (q_high - q_low) * 0.05
            x_min = max(0.0, float(q_low - pad))
            x_max = float(q_high + pad)
            axes[1, 0].set_xlim(x_min, x_max)

            excluded = int(len(volatility) - len(focus))
            if excluded > 0:
                axes[1, 0].text(
                    0.98,
                    0.95,
                    f"Outliers excluded: {excluded}",
                    ha="right",
                    va="top",
                    transform=axes[1, 0].transAxes,
                    fontsize=9,
                )


    avg_volume = ok_df["avg_volume"].dropna()
    positive_volume = avg_volume[avg_volume > 0]

    if positive_volume.empty:
        axes[1, 1].text(0.5, 0.5, "No positive avg volume data", ha="center", va="center", transform=axes[1, 1].transAxes)
    else:
        vmin = float(positive_volume.min())
        vmax = float(positive_volume.max())

        if np.isclose(vmin, vmax):
            bins = np.array([vmin * 0.9, vmax * 1.1])
        else:
            bins = np.logspace(np.log10(vmin), np.log10(vmax), 30)

        axes[1, 1].hist(positive_volume, bins=bins, color="#e76f51", alpha=0.8)
        axes[1, 1].set_xscale("log")

    axes[1, 1].set_title("Average Volume Distribution")
    axes[1, 1].set_xlabel("Average Volume (log scale)")
    axes[1, 1].set_ylabel("Frequency")

    plt.tight_layout()
    plt.show()


def plot_top_bottom_movers(ok_df: pd.DataFrame, top_n: int) -> None:
    movers = ok_df[["ticker", "total_return_pct"]].dropna().sort_values("total_return_pct")
    bottom = movers.head(top_n)
    top = movers.tail(top_n)

    fig, axes = plt.subplots(1, 2, figsize=RANKED_FIGSIZE)
    _plot_ranked_bars(
        axes[0],
        bottom["ticker"],
        bottom["total_return_pct"],
        title=f"Bottom {top_n} Tickers by Total Return",
        ylabel="Total Return %",
        color="#c1121f",
    )
    _plot_ranked_bars(
        axes[1],
        top["ticker"],
        top["total_return_pct"],
        title=f"Top {top_n} Tickers by Total Return",
        ylabel="Total Return %",
        color="#2a9d8f",
    )

    plt.tight_layout()
    plt.show()


def plot_all_total_returns(ok_df: pd.DataFrame) -> None:
    all_df = ok_df[["ticker", "total_return_pct"]].dropna().sort_values("total_return_pct", ascending=False).reset_index(drop=True)
    if all_df.empty:
        print("No total return data available for all-ticker plot.")
        return

    x = np.arange(len(all_df))
    colors = np.where(all_df["total_return_pct"] < 0, "#c1121f", "#2a9d8f")

    fig, ax = plt.subplots(figsize=(max(18, len(all_df) * 0.15), 7))
    ax.bar(x, all_df["total_return_pct"], color=colors, width=0.9)
    ax.axhline(0, color="#555555", linestyle="--", linewidth=1)

    step = max(1, len(all_df) // 40)
    ax.set_xticks(x[::step])
    ax.set_xticklabels(all_df["ticker"].iloc[::step], rotation=70)

    ax.set_title("Total Return for All Tickers")
    ax.set_xlabel("Ticker")
    ax.set_ylabel("Total Return %")

    plt.tight_layout()
    plt.show()


def top_return_performers(ok_df: pd.DataFrame, top_n: int) -> pd.DataFrame:
    return (
        ok_df[RESULT_COLUMNS]
        .sort_values("total_return_pct", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )


def worst_return_performers(ok_df: pd.DataFrame, top_n: int) -> pd.DataFrame:
    return (
        ok_df[RESULT_COLUMNS]
        .sort_values("total_return_pct", ascending=True)
        .head(top_n)
        .reset_index(drop=True)
    )


PCT_COLUMNS = ("total_return_pct", "daily_volatility_pct")


def _format_pct(series: pd.Series) -> pd.Series:
    return series.map(lambda x: f"{x:.2f}%" if pd.notna(x) else "")


def format_pct_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in PCT_COLUMNS:
        if col in out.columns:
            out[col] = _format_pct(out[col])
    return out


## Cell 5: Build Analysis Table
Builds the analysis table directly from tickers discovered in `dataset/stocks` and parsed CSV summaries.


In [ ]:
STOCKS_DIR = resolve_stocks_dir(PROJECT_ROOT)
TICKERS, price_paths = load_source_tickers(STOCKS_DIR)
ticker_df = pd.DataFrame({"ticker": TICKERS})

quality_df = build_quality_frame(price_paths)
analysis_df = build_analysis_frame(ticker_df, quality_df)

source_summary = pd.Series(
    {
        "source_directory": str(STOCKS_DIR),
        "csv_files_detected": len(price_paths),
        "unique_tickers_detected": len(TICKERS),
    }
)

display(source_summary.to_frame("value"))
display(count_table(analysis_df["status"], "status"))
display(format_pct_columns(analysis_df[PREVIEW_COLUMNS].head(MAX_PREVIEW_ROWS)))


## Cell 6: Final Visual Analytics
Builds overview charts, top/bottom movers, and displays strongest tickers by return from valid source-folder data.


In [ ]:
ok_df = analysis_df.loc[analysis_df["status"] == STATUS_OK].copy()

if ok_df.empty:
    print("No valid CSV data available for plotting from the source folder yet.")
else:
    plot_overview_charts(ok_df)

    plot_top_bottom_movers(ok_df, TOP_N)
    plot_all_total_returns(ok_df)

    print(f"Top {TOP_N} performers by total return")
    display(format_pct_columns(top_return_performers(ok_df, TOP_N)))

    print(f"Worst {TOP_N} performers by total return")
    display(format_pct_columns(worst_return_performers(ok_df, TOP_N)))


In [ ]:
VOL_MIN = 1.0
VOL_MAX = 3.0
LOG10_VOL_MIN = 5.9
LOG10_VOL_MAX = 6.1

filtered_tickers = (
    analysis_df.loc[analysis_df["status"] == STATUS_OK, ["ticker", "daily_volatility_pct", "avg_volume"]]
    .dropna(subset=["daily_volatility_pct", "avg_volume"])
    .assign(log10_avg_volume=lambda d: np.log10(d["avg_volume"]))
    .loc[lambda d: d["daily_volatility_pct"].between(VOL_MIN, VOL_MAX)]
    .loc[lambda d: d["log10_avg_volume"].between(LOG10_VOL_MIN, LOG10_VOL_MAX)]
    .sort_values(["daily_volatility_pct", "log10_avg_volume", "ticker"])
    .reset_index(drop=True)
)

print(
    f"Tickers with daily volatility in [{VOL_MIN}, {VOL_MAX}] and log10(avg_volume) in [{LOG10_VOL_MIN}, {LOG10_VOL_MAX}]: {len(filtered_tickers)}"
)
display(format_pct_columns(filtered_tickers))

print("\nTickers (line by line):")
if filtered_tickers.empty:
    print("No tickers matched the filter.")
else:
    for ticker in filtered_tickers["ticker"]:
        print(ticker)



In [ ]:
# Full success-rate heatmap (no threshold filter)

HEATMAP_HORIZON_BARS = 15
HEATMAP_STOP_LOSS_PCT_VALUES = np.linspace(0.10, 5.00, 20)          # x% below entry
HEATMAP_PROFIT_THRESHOLD_PCT_VALUES = np.linspace(0.10, 5.00, 20)   # x% above entry


def _success_counts_h0(df: pd.DataFrame, stop_loss_values: np.ndarray, profit_threshold_values: np.ndarray) -> tuple[np.ndarray, int]:
    """Vectorized success counts for horizon=0.

    Success definition matches label logic: target hit and stop NOT hit on the same bar.
    """
    numeric = df[["Open", "High", "Low"]].dropna().copy()
    if numeric.empty:
        return np.zeros((len(stop_loss_values), len(profit_threshold_values)), dtype=int), 0

    open_ = numeric["Open"].to_numpy(dtype=np.float64)
    high = numeric["High"].to_numpy(dtype=np.float64)
    low = numeric["Low"].to_numpy(dtype=np.float64)

    target_hits = high[:, None] >= (open_[:, None] * (1.0 + profit_threshold_values[None, :]))
    stop_hits = low[:, None] <= (open_[:, None] * (1.0 + stop_loss_values[None, :]))

    success_counts = (np.logical_not(stop_hits).astype(np.int32).T @ target_hits.astype(np.int32)).astype(int)
    return success_counts, int(open_.shape[0])


def _success_counts_general(
    df: pd.DataFrame,
    stop_loss_values: np.ndarray,
    profit_threshold_values: np.ndarray,
    horizon_bars: int,
) -> tuple[np.ndarray, int]:
    """General success counts for horizon>0 mirroring feature_builder._build_labels semantics."""
    numeric = df[["Open", "High", "Low"]].dropna().copy()
    if numeric.empty:
        return np.zeros((len(stop_loss_values), len(profit_threshold_values)), dtype=int), 0

    open_ = numeric["Open"].to_numpy(dtype=np.float64)
    high = numeric["High"].to_numpy(dtype=np.float64)
    low = numeric["Low"].to_numpy(dtype=np.float64)
    valid_trials = max(0, len(numeric) - int(horizon_bars))

    success_counts = np.zeros((len(stop_loss_values), len(profit_threshold_values)), dtype=int)
    if valid_trials <= 0:
        return success_counts, 0

    for s_idx, stop_loss in enumerate(stop_loss_values):
        stop_price = open_ * (1.0 + stop_loss)
        for p_idx, profit_threshold in enumerate(profit_threshold_values):
            target_price = open_ * (1.0 + profit_threshold)
            wins = 0

            for t in range(valid_trials):
                profit_hit = False
                stop_hit = False

                for k in range(horizon_bars + 1):
                    idx = t + k
                    if high[idx] >= target_price[t] and low[idx] <= stop_price[t]:
                        profit_hit = True
                        stop_hit = True
                        break
                    if high[idx] >= target_price[t]:
                        profit_hit = True
                        break
                    if low[idx] <= stop_price[t]:
                        stop_hit = True
                        break

                if profit_hit and not stop_hit:
                    wins += 1

            success_counts[s_idx, p_idx] = wins

    return success_counts, valid_trials


def compute_success_rate_heatmap_data(
    paths: list[Path],
    stop_loss_pct_values: np.ndarray,
    profit_threshold_pct_values: np.ndarray,
    horizon_bars: int,
) -> pd.DataFrame:
    stop_loss_values = -np.asarray(stop_loss_pct_values, dtype=float) / 100.0
    profit_threshold_values = np.asarray(profit_threshold_pct_values, dtype=float) / 100.0

    rows = []
    for path in paths:
        df = load_local_price_csv(path)
        ticker = path.stem.upper()

        if {"Open", "High", "Low"}.difference(df.columns):
            continue

        if horizon_bars == 0:
            success_counts, valid_trials = _success_counts_h0(df, stop_loss_values, profit_threshold_values)
        else:
            success_counts, valid_trials = _success_counts_general(
                df=df,
                stop_loss_values=stop_loss_values,
                profit_threshold_values=profit_threshold_values,
                horizon_bars=horizon_bars,
            )

        if valid_trials <= 0:
            continue

        success_rate_pct = (success_counts / valid_trials) * 100.0

        for s_idx, stop_loss_pct in enumerate(stop_loss_pct_values):
            for p_idx, profit_threshold_pct in enumerate(profit_threshold_pct_values):
                rows.append(
                    {
                        "ticker": ticker,
                        "stop_loss_pct": float(stop_loss_pct),
                        "profit_threshold_pct": float(profit_threshold_pct),
                        "success_rate_pct": float(success_rate_pct[s_idx, p_idx]),
                        "success_count": int(success_counts[s_idx, p_idx]),
                        "valid_trials": int(valid_trials),
                    }
                )

    return pd.DataFrame(rows)


def _plot_success_rate_heatmap(heat: pd.DataFrame, title: str) -> None:
    if heat.empty:
        print("No data to plot.")
        return

    fig_w = max(10, int(0.50 * heat.shape[1]) + 4)
    fig_h = max(7, int(0.40 * heat.shape[0]) + 3)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    im = ax.imshow(
        heat.to_numpy(dtype=float),
        aspect="auto",
        origin="lower",
        cmap="viridis",
        interpolation="nearest",
        vmin=0,
        vmax=100,
    )

    x_step = max(1, heat.shape[1] // 12)
    y_step = max(1, heat.shape[0] // 10)

    ax.set_xticks(np.arange(0, heat.shape[1], x_step))
    ax.set_xticklabels([f"{v:.2f}" for v in heat.columns.to_numpy()[::x_step]], rotation=45, ha="right")
    ax.set_yticks(np.arange(0, heat.shape[0], y_step))
    ax.set_yticklabels([f"{v:.2f}" for v in heat.index.to_numpy()[::y_step]])

    ax.set_xlabel("Profit threshold (%)")
    ax.set_ylabel("Stop loss below entry (%)")
    ax.set_title(title)

    cbar = fig.colorbar(im, ax=ax, pad=0.01)
    cbar.set_label("Success rate (%)")

    plt.tight_layout()
    plt.show()


success_rate_df = compute_success_rate_heatmap_data(
    paths=price_paths,
    stop_loss_pct_values=HEATMAP_STOP_LOSS_PCT_VALUES,
    profit_threshold_pct_values=HEATMAP_PROFIT_THRESHOLD_PCT_VALUES,
    horizon_bars=HEATMAP_HORIZON_BARS,
)

print(
    f"success_rate_df rows: {len(success_rate_df)} | tickers: {success_rate_df['ticker'].nunique() if not success_rate_df.empty else 0}"
)
display(success_rate_df.head(MAX_PREVIEW_ROWS))

if success_rate_df.empty:
    print("No success-rate data available.")
else:
    # 1) Aggregate heatmap across all tickers (mean success rate)
    mean_heat = (
        success_rate_df.groupby(["stop_loss_pct", "profit_threshold_pct"], as_index=False)["success_rate_pct"]
        .mean()
        .pivot(index="stop_loss_pct", columns="profit_threshold_pct", values="success_rate_pct")
        .sort_index(ascending=True)
    )
    _plot_success_rate_heatmap(
        mean_heat,
        title=(
            "Mean Success Rate Across All Tickers (No Threshold Filter)\n"
            f"horizon={HEATMAP_HORIZON_BARS} bars"
        ),
    )
